# Pinky 후방 USB 카메라 ChArUco 캘리브레이션

후방 USB 카메라를 실제 운용 설정(고정 by-id, 640×480, YUYV)으로 촬영하고 ROS CameraInfo YAML을 생성합니다.

현재 보드 설정:

- 배열: **7 × 5 squares**
- Dictionary: **DICT_4X4_50**
- 큰 체크 칸 한 변: **36.5 mm**
- 검은 ArUco 외곽 한 변: **26.0 mm**
- 예상 ArUco 마커: 17개
- 예상 ChArUco 코너: 24개

촬영 전 후방 카메라 ROS 퍼블리셔를 종료하세요. 같은 V4L2 장치는 두 프로세스가 동시에 열 수 없습니다.

## 1. 라이브러리 확인

`cv2.aruco`가 없다면 일반 `opencv-python` 대신 ArUco 모듈이 포함된 OpenCV 패키지가 필요합니다.

In [ ]:
from pathlib import Path
import json
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)
assert hasattr(cv2, "aruco"), "cv2.aruco가 없습니다. opencv-contrib 패키지를 설치하세요."
assert hasattr(cv2.aruco, "CharucoDetector"), "OpenCV의 CharucoDetector API가 필요합니다."
assert hasattr(cv2.aruco.CharucoBoard, "matchImagePoints"), "CharucoBoard.matchImagePoints API가 필요합니다."
assert hasattr(cv2, "calibrateCamera"), "cv2.calibrateCamera API가 필요합니다."
print("ChArUco 검출 및 표준 카메라 보정 API 확인 완료")

## 2. 보드·로봇·카메라 설정

`ROBOT_NAME`은 물리 장비 식별자입니다. Pinky1에서는 `pinky_6294`로 변경하세요. 보정 해상도와 픽셀 형식은 실제 후방 퍼블리셔 설정과 동일하게 유지합니다.

In [ ]:
SQUARES_X = 7
SQUARES_Y = 5
SQUARE_LENGTH = 0.0365  # 36.5 mm
MARKER_LENGTH = 0.0260  # 26.0 mm, 검은 ArUco 외곽 전체
DICTIONARY_ID = cv2.aruco.DICT_4X4_50
DICTIONARY_NAME = "DICT_4X4_50"

ROBOT_NAME = "pinky_15e2"  # Pinky1은 "pinky_6294"
CAMERA_NAME = "rear_camera"
VIDEO_DEVICE = Path(
    "/dev/v4l/by-id/"
    "usb-Jieli_Technology_USB_Composite_Device-video-index0"
)
IMAGE_WIDTH = 640
IMAGE_HEIGHT = 480
PIXEL_FORMAT = "YUYV"

CALIB_IMAGES_DIR = Path("datasets") / ROBOT_NAME / CAMERA_NAME
OUTPUT_DIR = Path("calibration_output") / ROBOT_NAME / CAMERA_NAME
CALIB_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dictionary = cv2.aruco.getPredefinedDictionary(DICTIONARY_ID)
board = cv2.aruco.CharucoBoard(
    (SQUARES_X, SQUARES_Y),
    SQUARE_LENGTH,
    MARKER_LENGTH,
    dictionary,
)
detector = cv2.aruco.CharucoDetector(board)

print("촬영 장치:", VIDEO_DEVICE)
print("이미지 폴더:", CALIB_IMAGES_DIR.resolve())
print("결과 폴더:", OUTPUT_DIR.resolve())

## 3. 보드 패턴 확인

아래 이미지는 dictionary와 배열 확인용입니다. 실제 캘리브레이션에는 가지고 있는 실물 보드를 사용하세요.

In [ ]:
preview = board.generateImage((700, 980), marginSize=30, borderBits=1)
plt.figure(figsize=(5, 7))
plt.imshow(preview, cmap="gray")
plt.title(f"{SQUARES_X}x{SQUARES_Y} / {DICTIONARY_NAME}")
plt.axis("off")
plt.show()

## 4. 후방 USB 카메라로 촬영

`RUN_CAPTURE = True`로 변경한 뒤 실행하세요. `c`를 입력하면 저장하고 `q`를 입력하면 종료합니다.

20~40장을 권장합니다. 보드를 중앙뿐 아니라 네 모서리, 가까운 거리, 먼 거리, 여러 기울기로 촬영하세요. 장치가 busy라면 후방 카메라 ROS 퍼블리셔를 먼저 종료해야 합니다.

In [ ]:
RUN_CAPTURE = False

if not RUN_CAPTURE:
    print("촬영을 시작하려면 RUN_CAPTURE = True로 변경하고 다시 실행하세요.")
else:
    if not VIDEO_DEVICE.exists():
        raise RuntimeError(f"후방 카메라 장치가 없습니다: {VIDEO_DEVICE}")

    cap = cv2.VideoCapture(str(VIDEO_DEVICE), cv2.CAP_V4L2)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*PIXEL_FORMAT))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, IMAGE_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, IMAGE_HEIGHT)
    if not cap.isOpened():
        raise RuntimeError(
            f"후방 카메라를 열 수 없습니다: {VIDEO_DEVICE}. "
            "후방 카메라 ROS 퍼블리셔가 실행 중인지 확인하세요."
        )

    actual_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    actual_fourcc_value = int(cap.get(cv2.CAP_PROP_FOURCC))
    actual_fourcc = "".join(
        chr((actual_fourcc_value >> (8 * index)) & 0xFF) for index in range(4)
    )
    print(
        f"협상된 카메라 설정: {actual_width}x{actual_height}, "
        f"pixel_format={actual_fourcc!r}"
    )
    if (actual_width, actual_height) != (IMAGE_WIDTH, IMAGE_HEIGHT):
        cap.release()
        raise RuntimeError(
            "카메라 해상도 협상 실패: "
            f"요청={(IMAGE_WIDTH, IMAGE_HEIGHT)}, "
            f"실제={(actual_width, actual_height)}"
        )
    if actual_fourcc.strip("\x00") != PIXEL_FORMAT:
        cap.release()
        raise RuntimeError(
            f"픽셀 형식 협상 실패: 요청={PIXEL_FORMAT}, 실제={actual_fourcc!r}"
        )

    existing = sorted(CALIB_IMAGES_DIR.glob("rear_*.jpg"))
    file_number = len(existing) + 1

    try:
        while True:
            command = input("c: 촬영, q: 종료 > ").strip().lower()
            if command == "q":
                break
            if command != "c":
                continue

            frame = None
            ok = False
            for _ in range(5):
                ok, frame = cap.read()
            if not ok or frame is None:
                print("프레임 획득 실패")
                continue

            path = CALIB_IMAGES_DIR / f"rear_{file_number:03d}.jpg"
            cv2.imwrite(str(path), frame)
            print("저장:", path)

            plt.figure(figsize=(10, 6))
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.axis("off")
            plt.show()
            file_number += 1
    finally:
        cap.release()
        print("카메라 종료")

이미 다른 방법으로 촬영했다면 JPG/PNG 원본을 `calib_charuco/` 폴더에 복사하고 다음 단계부터 실행하면 됩니다.

## 5. ChArUco 코너 검출 및 이미지 선별

사진마다 최소 8개 코너를 요구합니다. 모든 사진의 해상도는 같아야 합니다.

In [ ]:
image_paths = sorted(
    list(CALIB_IMAGES_DIR.glob("*.jpg"))
    + list(CALIB_IMAGES_DIR.glob("*.jpeg"))
    + list(CALIB_IMAGES_DIR.glob("*.png"))
)

if not image_paths:
    raise RuntimeError(f"{CALIB_IMAGES_DIR.resolve()}에 캘리브레이션 사진이 없습니다.")

all_charuco_corners = []
all_charuco_ids = []
accepted_paths = []
rejected_paths = []
preview_images = []
image_size = None

for path in image_paths:
    image = cv2.imread(str(path))
    if image is None:
        rejected_paths.append((path, "읽기 실패"))
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    current_size = (gray.shape[1], gray.shape[0])
    if image_size is None:
        image_size = current_size
    elif current_size != image_size:
        rejected_paths.append((path, f"해상도 불일치: {current_size}"))
        continue

    charuco_corners, charuco_ids, marker_corners, marker_ids = detector.detectBoard(gray)
    corner_count = 0 if charuco_ids is None else len(charuco_ids)

    if charuco_ids is None or corner_count < 8:
        rejected_paths.append((path, f"코너 부족: {corner_count}"))
        continue

    all_charuco_corners.append(charuco_corners)
    all_charuco_ids.append(charuco_ids)
    accepted_paths.append(path)

    drawn = image.copy()
    cv2.aruco.drawDetectedCornersCharuco(drawn, charuco_corners, charuco_ids)
    preview_images.append((path.name, drawn, corner_count))

print(f"전체: {len(image_paths)}")
print(f"사용: {len(accepted_paths)}")
print(f"제외: {len(rejected_paths)}")
print("이미지 크기:", image_size)

for path, reason in rejected_paths:
    print("제외:", path.name, "-", reason)

if len(accepted_paths) < 10:
    raise RuntimeError("유효 이미지가 10장 미만입니다. 다양한 각도에서 추가 촬영하세요.")

In [ ]:
show_count = min(12, len(preview_images))
cols = 3
rows = math.ceil(show_count / cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, 4.5 * rows))
axes = np.atleast_1d(axes).reshape(-1)

for axis in axes:
    axis.axis("off")

for axis, (name, image, count) in zip(axes, preview_images[:show_count]):
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.set_title(f"{name} / corners={count}")
    axis.axis("off")

plt.tight_layout()
plt.show()

## 6. 카메라 내부 파라미터 계산

검출된 ChArUco 코너를 `board.matchImagePoints()`로 실제 보드 좌표와 대응시킨 뒤, OpenCV 표준 `calibrateCamera` 계열 API로 보정합니다. 이 방식은 `calibrateCameraCharuco`가 없는 OpenCV 빌드에서도 동작합니다.

In [ ]:
criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    100,
    1e-9,
)
# 640x480 데이터에서 k3는 불안정해지기 쉬우므로 0으로 고정해 과적합을 막습니다.
CALIBRATION_FLAGS = cv2.CALIB_FIX_K3

object_points_all = []
image_points_all = []

for corners, ids in zip(all_charuco_corners, all_charuco_ids):
    object_points, image_points = board.matchImagePoints(corners, ids)
    object_points_all.append(np.asarray(object_points, dtype=np.float32))
    image_points_all.append(np.asarray(image_points, dtype=np.float32))

if hasattr(cv2, "calibrateCameraExtended"):
    result = cv2.calibrateCameraExtended(
        object_points_all,
        image_points_all,
        image_size,
        None,
        None,
        flags=CALIBRATION_FLAGS,
        criteria=criteria,
    )
    (
        rms_error,
        camera_matrix,
        dist_coeffs,
        rvecs,
        tvecs,
        std_intrinsics,
        std_extrinsics,
        per_view_errors,
    ) = result
else:
    rms_error, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        object_points_all,
        image_points_all,
        image_size,
        None,
        None,
        flags=CALIBRATION_FLAGS,
        criteria=criteria,
    )
    std_intrinsics = None
    std_extrinsics = None
    per_view_errors = None

print(f"RMS reprojection error: {rms_error:.6f} px")
print("\nCamera matrix K:\n", camera_matrix)
print("\nDistortion coefficients D:\n", dist_coeffs.reshape(-1))

if rms_error < 0.5:
    print("\n판정: 좋음")
elif rms_error < 1.0:
    print("\n판정: 사용 가능하지만 개선 여지가 있습니다.")
else:
    print("\n판정: 오차가 큽니다. 흐린 사진과 고오차 사진을 제거하고 재촬영하세요.")

## 7. 사진별 재투영 오차 확인

오차가 유난히 큰 사진은 제외한 뒤 5~7단계를 다시 실행할 수 있습니다.

In [ ]:
error_rows = []

for path, corners, ids, rvec, tvec in zip(
    accepted_paths,
    all_charuco_corners,
    all_charuco_ids,
    rvecs,
    tvecs,
):
    object_points, image_points = board.matchImagePoints(corners, ids)
    projected, _ = cv2.projectPoints(
        object_points,
        rvec,
        tvec,
        camera_matrix,
        dist_coeffs,
    )
    image_points = np.asarray(image_points, dtype=np.float64).reshape(-1, 2)
    projected = projected.reshape(-1, 2)
    rmse = float(np.sqrt(np.mean(np.sum((image_points - projected) ** 2, axis=1))))
    error_rows.append((path.name, len(ids), rmse))

error_rows.sort(key=lambda row: row[2], reverse=True)
for name, count, error in error_rows:
    print(f"{name:30s} corners={count:2d} error={error:.4f} px")

## 8. NPZ, OpenCV YAML, ROS CameraInfo YAML 저장

In [ ]:
npz_path = OUTPUT_DIR / "camera_charuco_calibration.npz"
opencv_yaml_path = OUTPUT_DIR / "camera_charuco_opencv.yaml"
ros_yaml_path = OUTPUT_DIR / "camera_charuco_ros.yaml"
metadata_path = OUTPUT_DIR / "camera_charuco_metadata.json"

np.savez(
    npz_path,
    camera_matrix=camera_matrix,
    dist_coeffs=dist_coeffs,
    image_width=image_size[0],
    image_height=image_size[1],
    reprojection_error=rms_error,
)

storage = cv2.FileStorage(str(opencv_yaml_path), cv2.FILE_STORAGE_WRITE)
storage.write("image_width", image_size[0])
storage.write("image_height", image_size[1])
storage.write("camera_matrix", camera_matrix)
storage.write("distortion_coefficients", dist_coeffs)
storage.write("reprojection_error", float(rms_error))
storage.release()

d = dist_coeffs.reshape(-1).tolist()
k = camera_matrix.reshape(-1).tolist()
projection = [
    k[0], k[1], k[2], 0.0,
    k[3], k[4], k[5], 0.0,
    k[6], k[7], k[8], 0.0,
]

ros_yaml_text = f"""image_width: {image_size[0]}
image_height: {image_size[1]}
camera_name: {CAMERA_NAME}
camera_matrix:
  rows: 3
  cols: 3
  data: {k}
distortion_model: plumb_bob
distortion_coefficients:
  rows: 1
  cols: {len(d)}
  data: {d}
rectification_matrix:
  rows: 3
  cols: 3
  data: [1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]
projection_matrix:
  rows: 3
  cols: 4
  data: {projection}
"""
ros_yaml_path.write_text(ros_yaml_text, encoding="utf-8")

metadata = {
    "robot_name": ROBOT_NAME,
    "camera_name": CAMERA_NAME,
    "video_device": str(VIDEO_DEVICE),
    "pixel_format": PIXEL_FORMAT,
    "calibration_flags": ["CALIB_FIX_K3"],
    "dictionary": DICTIONARY_NAME,
    "squares_x": SQUARES_X,
    "squares_y": SQUARES_Y,
    "square_length_m": SQUARE_LENGTH,
    "marker_length_m": MARKER_LENGTH,
    "image_width": image_size[0],
    "image_height": image_size[1],
    "rms_reprojection_error_px": float(rms_error),
    "accepted_images": [str(path) for path in accepted_paths],
    "per_image_rmse_px": {
        name: error for name, _, error in error_rows
    },
}
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("저장 완료:")
for path in (npz_path, opencv_yaml_path, ros_yaml_path, metadata_path):
    print(" -", path.resolve())

## 9. 왜곡 보정 결과 확인

직선과 보드 외곽이 자연스럽게 펴지는지 확인하세요.

In [ ]:
test_path = accepted_paths[0]
test_image = cv2.imread(str(test_path))
h, w = test_image.shape[:2]

new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(
    camera_matrix,
    dist_coeffs,
    (w, h),
    1.0,
    (w, h),
)
undistorted = cv2.undistort(
    test_image,
    camera_matrix,
    dist_coeffs,
    None,
    new_camera_matrix,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB))
axes[0].set_title("원본")
axes[1].imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
axes[1].set_title("왜곡 보정")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

cv2.imwrite(str(OUTPUT_DIR / "undistorted_preview.jpg"), undistorted)

## 10. ChArUco 보드 거리 검증

보드를 카메라에서 알려진 거리에 놓고 아래 결과의 `tvec`과 비교하세요. `Z`는 카메라 광축 방향 거리이며 단위는 미터입니다.

In [ ]:
pose_image = cv2.imread(str(accepted_paths[0]))
pose_gray = cv2.cvtColor(pose_image, cv2.COLOR_BGR2GRAY)
pose_corners, pose_ids, _, _ = detector.detectBoard(pose_gray)

if pose_ids is None or len(pose_ids) < 4:
    print("자세 추정에 필요한 코너가 부족합니다.")
else:
    object_points, image_points = board.matchImagePoints(pose_corners, pose_ids)
    valid, rvec, tvec = cv2.solvePnP(
        object_points,
        image_points,
        camera_matrix,
        dist_coeffs,
    )
    if not valid:
        print("자세 추정 실패")
    else:
        distance = float(np.linalg.norm(tvec))
        print("tvec [m]:", tvec.reshape(-1))
        print(f"직선거리: {distance:.4f} m")
        cv2.drawFrameAxes(
            pose_image,
            camera_matrix,
            dist_coeffs,
            rvec,
            tvec,
            0.05,
        )
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(pose_image, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()